# D.R.O.N.A. — 03 Embedding Quality

Compare the two embedding models D.R.O.N.A. uses for dual-index retrieval:

- **BAAI/bge-small-en-v1.5** — curriculum text (general semantic)
- **TechWolf/JobBERT-v3** — career/job text (Decorte et al. 2025, arXiv:2507.21609)

We check which model better separates *career* queries from *curriculum* queries
by intra/inter-cluster cosine similarity. Models download on first use; the cell
skips gracefully if `sentence-transformers` or weights are unavailable.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np

from drona.evaluation.queries import C1_QUERIES

career_q = [q.query_text for q in C1_QUERIES if q.expected_relevance == 'career']
curric_q = [q.query_text for q in C1_QUERIES if q.expected_relevance == 'curriculum']
print(f'{len(career_q)} career queries, {len(curric_q)} curriculum queries')

In [ ]:
def cos(a, b):
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-9)
    return a @ b.T

def separation(model_name):
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(model_name)
    ec, ek = m.encode(career_q), m.encode(curric_q)
    intra = (cos(ec, ec).mean() + cos(ek, ek).mean()) / 2
    inter = cos(ec, ek).mean()
    return float(intra), float(inter), float(intra - inter)

results = {}
for name in ['BAAI/bge-small-en-v1.5', 'TechWolf/JobBERT-v3']:
    try:
        intra, inter, gap = separation(name)
        results[name] = {'intra_sim': intra, 'inter_sim': inter, 'separation': gap}
        print(f'{name}: intra={intra:.3f} inter={inter:.3f} separation={gap:+.3f}')
    except Exception as exc:
        print(f'{name}: skipped ({exc})')

**Interpretation:** higher *separation* (intra-cluster minus inter-cluster
similarity) means the model groups same-domain queries together and pushes
cross-domain queries apart — desirable for tier/domain-aware retrieval. JobBERT-v3
is expected to separate career queries best, which is why D.R.O.N.A. routes job
text to it and curriculum text to bge (the dual-index decision, C1).